In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))


In [ ]:
import torch
import json
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from notebooks.local.utils import get_paths, create_folders, download_file

PATHS    = get_paths()
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
THRESHOLDS = [0.05, 0.10, 0.20]

create_folders(PATHS)
print("Device:", DEVICE, "  fp16:", USE_FP16)


In [ ]:
def load_pretrained(backbone, paths, device, use_fp16=False):
    """Load a pretrained backbone. Returns (model, img_size, patch_size)."""
    if backbone == 'dinov2':
        from src.models.dinov2.dinov2.models.vision_transformer import vit_base
        model = vit_base(
            img_size=(518, 518), patch_size=14,
            num_register_tokens=0, block_chunks=0, init_values=1.0,
        )
        ckpt = torch.load(paths['dinov2_w'], map_location=device, weights_only=True)
        model.load_state_dict(ckpt, strict=True)
        img_size, patch_size = 518, 14

    elif backbone == 'dinov3':
        from src.models.dinov3.dinov3.models.vision_transformer import vit_base as vit_base_v3
        model = vit_base_v3(img_size=512, patch_size=16)
        ckpt = torch.load(paths['dinov3_w'], map_location=device, weights_only=True)
        model.load_state_dict(ckpt, strict=True)
        img_size, patch_size = 512, 16

    elif backbone == 'sam':
        from src.models.segment_anything.segment_anything import sam_model_registry
        model = sam_model_registry['vit_b'](checkpoint=paths['sam_w'])
        img_size, patch_size = 512, 16

    else:
        raise ValueError(f"Unknown backbone: {backbone}")

    model = model.to(device)
    if use_fp16 and backbone != 'sam':
        model = model.half()
    model.eval()
    return model, img_size, patch_size


In [ ]:
from src.datasets.spair_dataset import SPairDataset

pair_ann = os.path.join(PATHS['spair71k'], 'PairAnnotation')
layout   = os.path.join(PATHS['spair71k'], 'Layout')
images   = os.path.join(PATHS['spair71k'], 'JPEGImages')

test_dataset = SPairDataset(pair_ann, layout, images, 'large', 0.1, 'test')
print(f"SPair-71k test pairs: {len(test_dataset)}")


## DINOv2 — Zero-shot Baseline

In [ ]:
from experiments.evaluate import evaluate, save_results

out_dir = os.path.join(PATHS['step1'], 'dinov2_argmax')
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(os.path.join(out_dir, 'overall_stats.json')):
    print("DINOv2 argmax — already done, loading results.")
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov2_argmax = json.load(f)
    print(dinov2_argmax)
else:
    model, img_size, patch_size = load_pretrained('dinov2', PATHS, DEVICE, USE_FP16)
    per_img, all_kp, t = evaluate(model, test_dataset, DEVICE, THRESHOLDS,
                                   use_windowed_softargmax=False)
    save_results(per_img, all_kp, out_dir, t, THRESHOLDS)
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov2_argmax = json.load(f)
    del model
    torch.cuda.empty_cache()
    print("DINOv2 argmax done.")


In [ ]:
from experiments.evaluate import evaluate_multilayer, save_results

out_dir = os.path.join(PATHS['step1'], 'dinov2_multilayer')
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(os.path.join(out_dir, 'overall_stats.json')):
    print("DINOv2 multilayer — already done, loading results.")
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov2_ml = json.load(f)
    print(dinov2_ml)
else:
    model, img_size, patch_size = load_pretrained('dinov2', PATHS, DEVICE, USE_FP16)
    per_img, all_kp, t = evaluate_multilayer(
        model, test_dataset, DEVICE, THRESHOLDS,
        use_windowed_softargmax=False, n_last_layers=3,
        patch_size=patch_size, resized_size=img_size,
    )
    save_results(per_img, all_kp, out_dir, t, THRESHOLDS)
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov2_ml = json.load(f)
    del model
    torch.cuda.empty_cache()
    print("DINOv2 multilayer done.")


## DINOv3 — Zero-shot Baseline

In [ ]:
out_dir = os.path.join(PATHS['step1'], 'dinov3_argmax')
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(os.path.join(out_dir, 'overall_stats.json')):
    print("DINOv3 argmax — already done, loading results.")
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov3_argmax = json.load(f)
    print(dinov3_argmax)
else:
    model, img_size, patch_size = load_pretrained('dinov3', PATHS, DEVICE, USE_FP16)
    per_img, all_kp, t = evaluate(
        model, test_dataset, DEVICE, THRESHOLDS,
        use_windowed_softargmax=False,
    )
    save_results(per_img, all_kp, out_dir, t, THRESHOLDS)
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov3_argmax = json.load(f)
    del model
    torch.cuda.empty_cache()
    print("DINOv3 argmax done.")


In [ ]:
out_dir = os.path.join(PATHS['step1'], 'dinov3_multilayer')
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(os.path.join(out_dir, 'overall_stats.json')):
    print("DINOv3 multilayer — already done, loading results.")
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov3_ml = json.load(f)
    print(dinov3_ml)
else:
    model, img_size, patch_size = load_pretrained('dinov3', PATHS, DEVICE, USE_FP16)
    per_img, all_kp, t = evaluate_multilayer(
        model, test_dataset, DEVICE, THRESHOLDS,
        use_windowed_softargmax=False, n_last_layers=3,
        patch_size=patch_size, resized_size=img_size,
    )
    save_results(per_img, all_kp, out_dir, t, THRESHOLDS)
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        dinov3_ml = json.load(f)
    del model
    torch.cuda.empty_cache()
    print("DINOv3 multilayer done.")


## SAM — Zero-shot Baseline

In [ ]:
import time
import torch.nn.functional as F
from src.features.extractor import extract_dense_features_SAM, pixel_to_patch_coord, patch_to_pixel_coord
from src.matching.strategies import find_best_match_argmax
from src.metrics.pck import compute_pck_spair71k
from notebooks.local.utils import safe_cosine_similarity

out_dir = os.path.join(PATHS['step1'], 'sam_argmax')
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(os.path.join(out_dir, 'overall_stats.json')):
    print("SAM argmax — already done, loading results.")
    with open(os.path.join(out_dir, 'overall_stats.json')) as f:
        sam_argmax = json.load(f)
    print(sam_argmax)
else:
    model, img_size, patch_size = load_pretrained('sam', PATHS, DEVICE, USE_FP16)
    t0 = time.time()
    per_img, all_kp = [], []

    for idx, sample in enumerate(test_dataset):
        src_tensor = sample['src_img'].unsqueeze(0).to(DEVICE)
        tgt_tensor = sample['trg_img'].unsqueeze(0).to(DEVICE)

        src_original_size = (sample['src_imsize'][2], sample['src_imsize'][1])
        tgt_original_size = (sample['trg_imsize'][2], sample['trg_imsize'][1])

        src_feat = extract_dense_features_SAM(model, src_tensor, image_size=img_size)
        tgt_feat = extract_dense_features_SAM(model, tgt_tensor, image_size=img_size)
        _, H, W, D = tgt_feat.shape
        tgt_flat = tgt_feat.reshape(H * W, D)

        src_kps = sample['src_kps'].numpy()
        trg_kps = sample['trg_kps'].numpy()
        kps_ids = sample['kps_ids']
        trg_bbox = sample['trg_bbox']
        pred = []

        for i in range(src_kps.shape[0]):
            px, py = pixel_to_patch_coord(
                src_kps[i, 0], src_kps[i, 1], src_original_size, patch_size, img_size)
            src_feature = src_feat[0, py, px, :]
            sims = safe_cosine_similarity(src_feature, tgt_flat)
            mx, my = find_best_match_argmax(sims, W)
            rx, ry = patch_to_pixel_coord(mx, my, tgt_original_size, patch_size, img_size)
            pred.append([rx, ry])

        image_pcks = {}
        for thr in THRESHOLDS:
            pck, correct_mask, dists = compute_pck_spair71k(pred, trg_kps.tolist(), trg_bbox, thr)
            image_pcks[thr] = pck
            for kid, p, g, d, c in zip(kps_ids, pred, trg_kps.tolist(), dists, correct_mask):
                all_kp.append({'image_idx': idx, 'category': sample['category'],
                                'keypoint_id': kid, 'pred': p, 'gt': g,
                                'distance': d, 'correct_at_threshold': c, 'threshold': thr})
        per_img.append({'category': sample['category'], 'pck_scores': image_pcks,
                        'num_keypoints': src_kps.shape[0]})
        if (idx + 1) % 200 == 0:
            print(f"  {idx+1}/{len(test_dataset)}")

    elapsed = time.time() - t0
    stats = {"inference_time_sec": elapsed}
    for thr in THRESHOLDS:
        vals = [m['pck_scores'][thr] for m in per_img]
        stats[f"pck@{thr:.2f}"] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
        print(f"SAM PCK@{thr:.2f}: {np.mean(vals):.2f}%")

    with open(os.path.join(out_dir, 'overall_stats.json'), 'w') as f:
        json.dump(stats, f, indent=2)
    sam_argmax = stats
    del model
    torch.cuda.empty_cache()
    print("SAM argmax done.")


## Results Summary

In [ ]:
rows = []
for label, var_name, d in [
    ('DINOv2 argmax',     'dinov2_argmax', os.path.join(PATHS['step1'], 'dinov2_argmax')),
    ('DINOv2 multilayer', 'dinov2_ml',     os.path.join(PATHS['step1'], 'dinov2_multilayer')),
    ('DINOv3 argmax',     'dinov3_argmax', os.path.join(PATHS['step1'], 'dinov3_argmax')),
    ('DINOv3 multilayer', 'dinov3_ml',     os.path.join(PATHS['step1'], 'dinov3_multilayer')),
    ('SAM argmax',        'sam_argmax',    os.path.join(PATHS['step1'], 'sam_argmax')),
]:
    stats_path = os.path.join(d, 'overall_stats.json')
    if os.path.exists(stats_path):
        with open(stats_path) as f:
            s = json.load(f)
        rows.append({
            'Model': label,
            'PCK@0.05': round(s.get('pck@0.05', {}).get('mean', float('nan')), 2),
            'PCK@0.10': round(s.get('pck@0.10', {}).get('mean', float('nan')), 2),
            'PCK@0.20': round(s.get('pck@0.20', {}).get('mean', float('nan')), 2),
        })
    else:
        rows.append({'Model': label, 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})

df = pd.DataFrame(rows).set_index('Model')
print(df.to_string())
